In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path 
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from skimage import io, transform, color, filters, measure, feature, morphology
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage.segmentation import clear_border
from scipy import ndimage as ndi

from skimage.io import imread
from scipy.ndimage import gaussian_filter

!pip install tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array

!pip install opencv-python
import cv2

### Dataframe
- each row has corresponding MSI and HSI paths (1:1 image correspondence)

In [ ]:
root = Path("MSI_HSI_Kidney_Dataset")
rows = []

# create a dictionary for each image
for i in root.glob("**/*.jpg"):
    name = i.stem # retrieve file name (without suffix)
    mode, _, label, num = name.split("_") # split file name (e.g. hyperspectral_kidney_normal_0001)
    rows.append({
        "id": name,
        "mode": "hsi" if mode == "Hyperspectral" else "msi",
        "label": 1 if label == "tumor" else 0, 
        "num": num,
        "path": str(i)
    })

df = pd.DataFrame(rows) # create a DataFrame of all dictionaries

# merge HSI and MSI into one row
df_hsi = df[df["mode"] == "hsi"]
df_msi = df[df["mode"] == "msi"]
df_merged = df_hsi.merge(df_msi, on=["label", "num"], suffixes=("_hsi", "_msi"))

# verification
# print(df_merged.head(10))
# print(df_merged.tail(10))

### Preprocessing: Foreground Mask Edge Cases

In [ ]:
def test_preprocess_hsv_mask(path):
    
    # load image using tensorflow
    image_raw = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image_raw, channels=3).numpy()
    h, w = image.shape[:2]
    
    image_cropped = image[5:-5, 5:-5] # trims several pixels from edges to slightly reduce background artifacts
                                      # but the dimensions get restored when the dataset undergoes preprocesssing and is
                                      # saved at 128 x 128 pixels (original dataset: 512 x 512 pixels)
    
    # generate HSV mask (Hue, Saturation, Value)
    hsv = cv2.cvtColor(image_cropped, cv2.COLOR_RGB2HSV) # convert from RGB to HSV -> separates color info from brightness
                                                         # helpful because we want to detect the red background to move
                                                         # while ensuring that sample illumination conditions does not impact the red
                                                         # background (e.g. making the bg red more darker or lighter across samples
                                                         # makes it harder to detect)
                                                         # RGB might detect dark and light red as 2 diff colors; HSV does not 
    
    lower_red = np.array([0, 100, 20]) # [Hue, Saturation, Value]; "start" of red
    upper_red = np.array([15, 255, 255]) # [hue Saturation, Value]; "end" of red (almost orange)
    
    background_mask = cv2.inRange(hsv, lower_red, upper_red) # everything that falls in lower_red and upper_red is now white; everything else black
    foreground_mask = cv2.bitwise_not(background_mask) # INVERTS it (so everything that did fall in lower_red and upper_red is now BLACK)

    # safety implementation: in the case that the kidney scan actually extends to the edge of the image, set several
    # edge pixels to 0 so that in the next step, when it looks for the largest foreground mask region, it doesn't take
    # the background instead of the kidney scan itself
    margin = 4
    foreground_mask[0:margin, :] = 0
    foreground_mask[-margin:, :] = 0
    foreground_mask[:, 0:margin] = 0
    foreground_mask[:, -margin:] = 0

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(foreground_mask, connectivity=8) # find largest region of white area (scan)

    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        temp_mask = np.zeros_like(foreground_mask)
        temp_mask[labels == largest_label] = 255
        
        # different pathways for coronal and axial to maximize preprocessing for each scan type
        organ_w = stats[largest_label, cv2.CC_STAT_WIDTH] # get larget region's width
        organ_h = stats[largest_label, cv2.CC_STAT_HEIGHT] # get largest region's height
        organ_aspect_ratio = organ_h / organ_w
        
        points = cv2.findNonZero(temp_mask)
        if points is not None:
            pts = points.reshape(-1, 2)
            curr_h, curr_w = temp_mask.shape[:2]
            
            # coronal (front-back) scans
            if organ_aspect_ratio > 1.2: # if the largest region's height / width > 1.2 (meaning height > width)
                mid_x = curr_w // 2 
                left_pts = pts[pts[:, 0] < mid_x # left plane
                right_pts = pts[pts[:, 0] >= mid_x] # right plane

                # all this code is just saying to draw a "bounding box" connecting the top left to top right and
                # bottom left to bottom right points via 2 line segments to "patch" any holes in the middle
                # this is because the middle of the scan has some red parts that might be removed from the earlier
                # background removal attempt, where it labels anything that is red as the background

                # horizontal lines for filling, only at the top and bottom of coronal scans; NOT vertically
                if len(left_pts) > 0 and len(right_pts) > 0:  
                    lt_pt = left_pts[np.argmin(left_pts[:, 1])]
                    rt_pt = right_pts[np.argmin(right_pts[:, 1])]
                    cv2.line(temp_mask, tuple(lt_pt), tuple(rt_pt), 255, 5)
                    
                    lb_pt = left_pts[np.argmax(left_pts[:, 1])]
                    rb_pt = right_pts[np.argmax(right_pts[:, 1])]
                    cv2.line(temp_mask, tuple(lb_pt), tuple(rb_pt), 255, 5)
                    
                    rightmost_pt = right_pts[np.argmax(right_pts[:, 0])]
                    cv2.line(temp_mask, tuple(rt_pt), tuple(rightmost_pt), 255, 5)
                    cv2.line(temp_mask, tuple(rb_pt), tuple(rightmost_pt), 255, 5)

            # axial (top-bottom) scans
            else:
                hull = cv2.convexHull(pts) # finds smallest possible convex shape that contains every white region pixel contained the largest region
                cv2.drawContours(temp_mask, [hull], -1, 255, thickness=2) # draws "rubberband" outline onto mask

        # fill interior
        contours, _ = cv2.findContours(temp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            final_mask = np.zeros_like(temp_mask)
            big_cnt = max(contours, key=cv2.contourArea)
            cv2.drawContours(final_mask, [big_cnt], -1, 255, thickness=cv2.FILLED) # again, like the coronal one, fills the inside 
        else:
            final_mask = temp_mask
    else:
        final_mask = foreground_mask

    kernel = np.ones((5,5), np.uint8) 
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, kernel) # a final smoothing to reduce noise introduced from foreground mask extraction
                                                                       # use morphologyEx as a pixel polish; also rounds out any "jagged" boundaries
    
    masked_result = cv2.bitwise_and(image_cropped, image_cropped, mask=final_mask)
    
    return image_cropped, final_mask, masked_result

In [ ]:
# 1. Select a random row from your merged dataframe
# random_row = df_merged.sample(n=1).iloc[0]
random_row = df_merged[df_merged['path_msi'].str.contains('31') & df_merged['path_hsi'].str.contains('normal')].iloc[0]

# 2. Extract paths and label
test_path = random_row['path_hsi']
test_label = random_row['label']

# 3. Run the improved masking function
original, mask, masked_result = test_preprocess_hsv_mask(test_path)

# 4. Plot the comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(original)
axes[0].set_title(f"Original (Label: {test_label})")
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title("Largest Component Mask")
axes[1].axis('off')

axes[2].imshow(masked_result)
axes[2].set_title("Cleaned Model Input")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Testing random file: {test_path}")  